# 05 – XAI Analysis: SHAP-Erklärung der Gebotsentscheidung

Dieses Notebook erklärt **warum** das Modell ein bestimmtes Gebot abgibt —
nicht wie gut es prognostiziert.

## Fragestellung

Im Newsvendor-Rahmen ist das optimale Gebot das τ\*-Quantil der Prognosedistribution:

$$b_t = Q_{\tau^*}(\hat{F}_t), \quad \tau^* = \frac{c_{under}}{c_{under} + c_{over}} = 0{,}524$$

SHAP (Lundberg & Lee 2017) zerlegt dieses Gebot additiv über alle Features:

$$b_t = \phi_0 + \sum_{i=1}^{p} \phi_i(x_t)$$

## Methodik

| Modell | Explainer | Begründung |
|--------|-----------|------------|
| Elastic Net | Koeffizienten (exakt) | Lineares Modell, direkt interpretierbar |
| QGB Q0.5 | TreeExplainer (exakt) | GradientBoostingRegressor, Theorem 1 in Lundberg & Lee |
| QRF | TreeExplainer (exakt) | Erklärt E[y\|x] als Proxy für das Quantil |
| Random Forest | TreeExplainer (exakt) | Point-Forecast-Vergleich |
| Neural Net | KernelExplainer (approx.) | Modellunabhängig, Abschn. 3.3 in Lundberg & Lee |

## Zentrale Forschungsfrage

Sind die Feature-Wichtigkeiten für die Gebotsentscheidung über verschiedene Modellklassen hinweg konsistent?
Falls ja, ist die Erklärbarkeit **modell-robust** — ein interpretierbares Modell verzichtet auf keine Information.

## 0. Umgebungsprüfung

Python 3.12 mit sklearn ≥ 1.8 erforderlich (Modelle wurden damit gespeichert).

In [ ]:
import sys
import sklearn, shap
print(f"Python:  {sys.version.split()[0]}")
print(f"sklearn: {sklearn.__version__}")
print(f"shap:    {shap.__version__}")
assert sys.version_info >= (3, 12), "Falscher Kernel! Bitte Python 3.12 auswählen."
assert tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8),     f"sklearn zu alt: {sklearn.__version__} (brauche >= 1.8)"

## 1. Setup und Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai.shap_analysis import shap_qgb, shap_qrf, shap_neural_net, shap_random_forest

MODELS_DIR   = PROJECT_ROOT / "results" / "models"
FORECAST_DIR = PROJECT_ROOT / "results" / "forecasts"
TABLES_DIR   = PROJECT_ROOT / "results" / "tables"
FIGURES_DIR  = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# τ* aus Notebook 04
TAU_STAR = 0.5238
# Nächstes verfügbares Quantil im QGB-Modell (0.1, 0.25, 0.5, 0.75, 0.9)
TAU_QGB  = 0.5

print(f"Project root: {PROJECT_ROOT}")
print(f"τ* = {TAU_STAR:.4f}  →  QGB-Quantil: Q{TAU_QGB}")

## 2. Daten laden und Train/Test rekonstruieren

Wir replizieren den exakten Feature-Engineering-Schritt aus Notebook 03,
um den echten `X_train` für den KernelExplainer-Background zu erhalten.
Alle Lag- und Rolling-Features werden identisch berechnet.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "final_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

df_model = df.copy()
df_model["energy_mwh"] = df_model["power"].clip(lower=0) / 1000
df_model["hour"]       = df_model["timestamp"].dt.hour
df_model["dayofweek"]  = df_model["timestamp"].dt.dayofweek
df_model["month"]      = df_model["timestamp"].dt.month

df_model["hour_sin"] = np.sin(2 * np.pi * df_model["hour"] / 24)
df_model["hour_cos"] = np.cos(2 * np.pi * df_model["hour"] / 24)
df_model["dow_sin"]  = np.sin(2 * np.pi * df_model["dayofweek"] / 7)
df_model["dow_cos"]  = np.cos(2 * np.pi * df_model["dayofweek"] / 7)

for lag in [24, 48, 72, 168]:
    df_model[f"energy_lag_{lag}"] = df_model["energy_mwh"].shift(lag)

if "wind_speed" in df_model.columns:
    for lag in [24, 48, 72, 168]:
        df_model[f"wind_speed_lag_{lag}"] = df_model["wind_speed"].shift(lag)

df_model["energy_roll_mean_24"]  = df_model["energy_mwh"].shift(24).rolling(24).mean()
df_model["energy_roll_std_24"]   = df_model["energy_mwh"].shift(24).rolling(24).std()
df_model["energy_roll_mean_168"] = df_model["energy_mwh"].shift(24).rolling(168).mean()
df_model["energy_roll_std_168"]  = df_model["energy_mwh"].shift(24).rolling(168).std()

if "wind_speed" in df_model.columns:
    df_model["wind_roll_mean_24"]  = df_model["wind_speed"].shift(24).rolling(24).mean()
    df_model["wind_roll_std_24"]   = df_model["wind_speed"].shift(24).rolling(24).std()
    df_model["wind_roll_mean_168"] = df_model["wind_speed"].shift(24).rolling(168).mean()
    df_model["wind_roll_std_168"]  = df_model["wind_speed"].shift(24).rolling(168).std()

BASE_FEATURES = [
    "energy_lag_24", "energy_lag_48", "energy_lag_72", "energy_lag_168",
    "energy_roll_mean_24", "energy_roll_std_24", "energy_roll_mean_168", "energy_roll_std_168",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month",
]
WIND_FEATURES = [
    "wind_speed_lag_24", "wind_speed_lag_48", "wind_speed_lag_72", "wind_speed_lag_168",
    "wind_roll_mean_24", "wind_roll_std_24", "wind_roll_mean_168", "wind_roll_std_168",
]
FEATURES = BASE_FEATURES + [f for f in WIND_FEATURES if f in df_model.columns]
TARGET   = "energy_mwh"

df_model = df_model.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
X = df_model[FEATURES].to_numpy()

split_idx = int(len(df_model) * 0.8)
X_train = X[:split_idx]
X_test  = X[split_idx:]

print(f"Features: {len(FEATURES)}")
print(f"X_train:  {X_train.shape}")
print(f"X_test:   {X_test.shape}")

## 3. Modelle laden

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)

qgb_models = joblib.load(MODELS_DIR / "quantile_gradient_boosting_day_ahead.joblib")
qrf_model  = joblib.load(MODELS_DIR / "qrf_day_ahead.joblib")
rf_model   = joblib.load(MODELS_DIR / "random_forest_day_ahead.joblib")
en_model   = joblib.load(MODELS_DIR / "elastic_net_day_ahead.joblib")
nn_model   = joblib.load(MODELS_DIR / "neural_network_day_ahead.joblib")

# Kompatibilitätsprüfung Neural Net
try:
    n_nn_features = nn_model.named_steps["scaler"].n_features_in_
except Exception:
    n_nn_features = -1

if n_nn_features != len(FEATURES):
    print(f"nn_model hat {n_nn_features} Features statt {len(FEATURES)} -> trainiere neu ...")
    from src.models import neural_net as neural_net_module
    y = df_model[TARGET].to_numpy()
    nn_model = neural_net_module.train(
        X_train, y[:split_idx],
        hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42,
    )
    joblib.dump(nn_model, MODELS_DIR / "neural_network_day_ahead.joblib")
    print("  Gespeichert: neural_network_day_ahead.joblib (21 Features)")
    n_nn_features = len(FEATURES)

print(f"QGB Quantile:  {sorted(qgb_models.keys())}")
print(f"QRF:           {type(qrf_model).__name__}")
print(f"RF:            {type(rf_model).__name__}  ({rf_model.n_estimators} Bäume)")
print(f"Elastic Net:   {type(en_model).__name__}")
print(f"Neural Net:    {type(nn_model).__name__}  ({n_nn_features} Features)")

## 4. Elastic Net – Koeffizienten (interpretierbare Baseline)

Elastic Net ist ein lineares Modell: jeder Koeffizient gibt direkt an,
um wie viele MWh das Gebot steigt, wenn das Feature um eine Einheit zunimmt
(ceteris paribus). Features mit Koeffizient ≈ 0 wurden durch L1-Regularisierung eliminiert.

In [ ]:
coefs = pd.Series(
    en_model.named_steps["elastic_net"].coef_,
    index=FEATURES,
)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d62728" if c < 0 else "#1f77b4" for c in coefs.sort_values()]
coefs.sort_values().plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Elastic Net – Koeffizienten (Gebotsbeitrag)")
ax.set_xlabel("Koeffizient [MWh / Einheit]")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_en_coefs.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top-Features (Elastic Net):")
display(coefs.sort_values(ascending=False).rename("Koeffizient [MWh/Einheit]").to_frame())

## 5. SHAP – Quantile Gradient Boosting (τ\*-Modell)

Das QGB-Modell mit `alpha=0.5` (nächstes verfügbares Quantil zu τ\*=0.524) wird direkt erklärt.
TreeExplainer liefert **exakte** SHAP-Werte in polynomialer Laufzeit (Theorem 1, Lundberg & Lee 2017).

Die SHAP-Werte messen die Abweichung des Gebots vom Erwartungswert $\mathbb{E}[Q_{0.5}]$,
aufgeteilt auf einzelne Features.

In [ ]:
shap_vals_qgb = shap_qgb(
    qgb_models, tau_star=TAU_QGB, X_test=X_test[:500], feature_names=FEATURES,
)

print(f"SHAP-Matrix Shape: {shap_vals_qgb.shape}")
mean_abs_qgb = pd.Series(np.abs(shap_vals_qgb).mean(axis=0), index=FEATURES, name="|SHAP| QGB")
print("\nMittlerer |SHAP| pro Feature:")
display(mean_abs_qgb.sort_values(ascending=False).to_frame())

## 6. SHAP – Quantile Regression Forest (E[y|X]-Proxy)

Der QRF nutzt intern einen `RandomForestRegressor` (MSE-Training).
TreeExplainer erklärt daher $\mathbb{E}[y|x]$ als Proxy für die Quantilattribution.
Die Feature-Rangfolge bleibt gültig, solange die Quantile monoton von $\mathbb{E}[y|x]$ abhängen.

In [ ]:
shap_vals_qrf = shap_qrf(
    qrf_model, X_test=X_test[:500], feature_names=FEATURES,
)

print(f"SHAP-Matrix Shape: {shap_vals_qrf.shape}")
mean_abs_qrf = pd.Series(np.abs(shap_vals_qrf).mean(axis=0), index=FEATURES, name="|SHAP| QRF")
print("\nMittlerer |SHAP| pro Feature:")
display(mean_abs_qrf.sort_values(ascending=False).to_frame())

## 7. SHAP – Neural Net (KernelExplainer)

KernelExplainer ist modellunabhängig (Abschn. 3.3, Lundberg & Lee 2017).
Er approximiert SHAP via gewichteter linearer Regression über Feature-Koalitionen.

- **Background:** 50 k-Means-Cluster aus X_train (repräsentiert $\mathbb{E}[f(X)]$)
- **nsamples=300:** stabile Schätzung (bei 21 Features: $2^{21}$ mögliche Koalitionen → Sampling)
- **Hinweis:** Läuft einige Minuten (100 Testpunkte × 300 Samples).

In [ ]:
shap_vals_nn = shap_neural_net(
    nn_model, X_test=X_test[:100], X_train=X_train,
    feature_names=FEATURES, n_background=50, nsamples=300,
)

shap_arr_nn = np.array(shap_vals_nn)
print(f"SHAP-Matrix Shape: {shap_arr_nn.shape}")
mean_abs_nn = pd.Series(np.abs(shap_arr_nn).mean(axis=0), index=FEATURES, name="|SHAP| NN")
print("\nMittlerer |SHAP| pro Feature:")
display(mean_abs_nn.sort_values(ascending=False).to_frame())

## 8. SHAP – Random Forest (Point Forecast, Vergleich)

Der Point-Forecast-RF bietet $\hat{y}$ direkt als Gebot (kein Quantil-Tuning).
Er dient zum modellübergreifenden Vergleich der Feature-Wichtigkeiten.

In [ ]:
shap_vals_rf = shap_random_forest(
    rf_model, X_test=X_test[:500], feature_names=FEATURES,
)

print(f"SHAP-Matrix Shape: {shap_vals_rf.shape}")
mean_abs_rf = pd.Series(np.abs(shap_vals_rf).mean(axis=0), index=FEATURES, name="|SHAP| RF")
print("\nMittlerer |SHAP| pro Feature:")
display(mean_abs_rf.sort_values(ascending=False).to_frame())

## 9. Vergleich: Feature-Rangfolge über alle Modelle

Wenn alle Modelle dieselben Top-Features identifizieren,
ist die Erklärung der Gebotsentscheidung **robust gegenüber der Modellwahl**.

In [ ]:
importance_df = pd.DataFrame({
    "QGB (tau*=Q0.5)":    pd.Series(np.abs(shap_vals_qgb).mean(axis=0), index=FEATURES),
    "QRF (E[y|X]-Proxy)": pd.Series(np.abs(shap_vals_qrf).mean(axis=0), index=FEATURES),
    "RF (Point)":          pd.Series(np.abs(shap_vals_rf).mean(axis=0),  index=FEATURES),
    "Neural Net":          pd.Series(np.abs(shap_arr_nn).mean(axis=0),   index=FEATURES),
}).sort_values("QGB (tau*=Q0.5)", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
importance_df.plot(kind="barh", ax=ax, width=0.7)
ax.set_xlabel("Mittlerer |SHAP|-Wert [MWh]")
ax.set_title("Feature-Wichtigkeit im Vergleich - Alle Modelle")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_feature_importance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nSpearman-Rangkorrelation der Feature-Wichtigkeiten:")
from scipy.stats import spearmanr
cols = importance_df.columns.tolist()
corr_rows = []
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        rho, pval = spearmanr(importance_df[c1], importance_df[c2])
        corr_rows.append({"Modell A": c1, "Modell B": c2, "Spearman rho": round(rho, 3), "p-Wert": round(pval, 4)})
display(pd.DataFrame(corr_rows))

## 10. Einzelbeispiel: SHAP Waterfall Plot (QGB)

Für einen einzelnen Zeitpunkt zeigt der Waterfall Plot, welche Features
das Gebot nach oben oder unten treiben.
Gewählt wird der Testpunkt mit dem größten SHAP-Gesamtbetrag
(d.h. die am stärksten vom Erwartungswert abweichende Gebotssituation).

In [ ]:
import shap

total_abs   = np.abs(shap_vals_qgb).sum(axis=1)
idx         = int(np.argmax(total_abs))
explainer_qgb = shap.TreeExplainer(qgb_models[TAU_QGB])

exp = shap.Explanation(
    values=shap_vals_qgb[idx],
    base_values=explainer_qgb.expected_value,
    data=X_test[:500][idx],
    feature_names=FEATURES,
)

shap.waterfall_plot(exp, show=False)
plt.title(f"Waterfall – QGB Q{TAU_QGB} – Testpunkt #{idx}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "shap_qgb_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Erwartungswert E[f(X)] = {explainer_qgb.expected_value:.3f} MWh")
print(f"Gebot fuer Testpunkt #{idx}: {shap_vals_qgb[idx].sum() + explainer_qgb.expected_value:.3f} MWh")
print(f"Feature-Werte:")
for fname, fval, sval in sorted(
    zip(FEATURES, X_test[:500][idx], shap_vals_qgb[idx]),
    key=lambda x: abs(x[2]), reverse=True
)[:8]:
    print(f"  {fname:<25} = {fval:7.3f}   SHAP: {sval:+.3f}")

## 11. Zusammenfassung

In [ ]:
print("=" * 65)
print("XAI ANALYSIS – ZUSAMMENFASSUNG")
print("=" * 65)

# Top-3 jedes Modells
summary_data = {
    "QGB (tau*=Q0.5)":    pd.Series(np.abs(shap_vals_qgb).mean(axis=0), index=FEATURES),
    "QRF (E[y|X]-Proxy)": pd.Series(np.abs(shap_vals_qrf).mean(axis=0), index=FEATURES),
    "RF (Point)":          pd.Series(np.abs(shap_vals_rf).mean(axis=0),  index=FEATURES),
    "Neural Net":          pd.Series(np.abs(shap_arr_nn).mean(axis=0),   index=FEATURES),
}
print("\nTop-5 Features pro Modell:")
top5 = pd.DataFrame({k: v.sort_values(ascending=False).head(5).index.tolist()
                     for k, v in summary_data.items()},
                    index=[f"Rang {i+1}" for i in range(5)])
display(top5)

# Verbindung zu NB04: Lade Bidding-Ergebnisse
try:
    bidding = pd.read_csv(TABLES_DIR / "bidding_results_day_ahead.csv")
    best_bid = bidding[bidding["model"] != "Oracle (perfekt)"].iloc[0]
    print(f"\nBestes Bidding-Modell (NB04): {best_bid['model']}")
    print(f"  Mean NV Loss: {best_bid['mean_nv_loss']:.4f} EUR/h")
    print(f"  -> Dieses Modell wird in Abschnitt 5 (SHAP-QGB) erklaert.")
except FileNotFoundError:
    print("\n(bidding_results_day_ahead.csv nicht gefunden - erst NB04 ausfuehren)")

print("\nGespeicherte Abbildungen:")
for f in sorted(FIGURES_DIR.glob("shap_*.png")):
    print(f"  {f.name}")